# Seeded Weekly Comparison (DQN vs DQN No-LLM)

This notebook summarizes 5 seeded weekly training runs with 10% stop-loss at inference.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

summary_path = 'test_results/seeded_weekly_full/seeded_weekly_summary.csv'
raw_path = 'test_results/seeded_weekly_full/seeded_weekly_runs.csv'
summary = pd.read_csv(summary_path)
raw = pd.read_csv(raw_path)
summary.head()


In [ ]:
def fmt(mean, std):
    return f"{mean:.2f} ± {std:.2f}"

ret = summary.pivot(index='ticker', columns='method', values=['return_mean_pct', 'return_std_pct'])
fin = summary.pivot(index='ticker', columns='method', values=['final_mean', 'final_std'])

rows = []
for ticker in sorted(summary['ticker'].unique()):
    row = {
        'Ticker': ticker,
        'DQN Return % (mean±std)': fmt(ret.loc[ticker, ('return_mean_pct', 'dqn')], ret.loc[ticker, ('return_std_pct', 'dqn')]),
        'DQN No-LLM Return % (mean±std)': fmt(ret.loc[ticker, ('return_mean_pct', 'dqn_nollm')], ret.loc[ticker, ('return_std_pct', 'dqn_nollm')]),
        'DQN Final $ (mean±std)': fmt(fin.loc[ticker, ('final_mean', 'dqn')], fin.loc[ticker, ('final_std', 'dqn')]),
        'DQN No-LLM Final $ (mean±std)': fmt(fin.loc[ticker, ('final_mean', 'dqn_nollm')], fin.loc[ticker, ('final_std', 'dqn_nollm')]),
    }
    rows.append(row)
display(pd.DataFrame(rows))


In [ ]:
tickers = sorted(summary['ticker'].unique())
x = np.arange(len(tickers))
width = 0.38

dqn = summary[summary['method']=='dqn'].set_index('ticker').loc[tickers]
nollm = summary[summary['method']=='dqn_nollm'].set_index('ticker').loc[tickers]

plt.figure(figsize=(14, 5))
plt.bar(x - width/2, dqn['return_mean_pct'], width, yerr=dqn['return_std_pct'], capsize=4, label='DQN')
plt.bar(x + width/2, nollm['return_mean_pct'], width, yerr=nollm['return_std_pct'], capsize=4, label='DQN No-LLM')
plt.xticks(x, tickers, rotation=45)
plt.ylabel('Return % (mean ± std)')
plt.title('Weekly Seeded Comparison by Ticker')
plt.legend()
plt.tight_layout()
plt.show()
